# 15 — Pipeline, persistência e operação

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 15/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/15_pipeline_reproduzivel.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_15_pipeline_reproduzivel.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Salvar artefatos somente em saídas
- Definir funções com entradas explícitas e evitar mutar dados originais

## Fundamentação conceitual
Uma pipeline reproduzível distingue fontes imutáveis, dados derivados, artefatos e métricas. Não sobrescreva `dados/ratings.csv`, especialmente ao simular novas avaliações.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Treinar baseline encapsulado

In [2]:
def fit_popularity(frame,reg=20):
    global_mean=frame.rating.mean()
    st=frame.groupby('movieId').rating.agg(['mean','count'])
    return (st['mean']*st['count']+reg*global_mean)/(st['count']+reg)
scores=fit_popularity(ratings);print(scores.sort_values(ascending=False).head())

movieId
318     4.373980
858     4.214770
2959    4.208114
260     4.177237
50      4.172014
dtype: float64


**Interpretação.** A função aceita um DataFrame, o que facilita testes com treino separado.

## Persistência em saída isolada

In [3]:
from pathlib import Path
out=Path('outputs');out.mkdir(exist_ok=True)
path=out/'ranking_popularidade.csv'
scores.rename('score').to_csv(path)
loaded=pd.read_csv(path)
assert len(loaded)==len(scores)
print('Artefato:',path,'linhas:',len(loaded))

Artefato: outputs/ranking_popularidade.csv linhas: 9724


**Interpretação.** CSV reproduzível evita desserialização insegura de pickle não confiável.

## Nova interação sem alterar origem

In [4]:
new=pd.DataFrame([{'userId':99999,'movieId':1,'rating':4.0,'timestamp':int(ratings.timestamp.max())+1}])
extended=pd.concat([ratings,new],ignore_index=True)
print('Fonte:',len(ratings),'Cópia ampliada:',len(extended));assert len(extended)==len(ratings)+1

Fonte: 100836 Cópia ampliada: 100837


**Interpretação.** Somente a cópia em memória recebe a avaliação simulada.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** reprodutibilidade. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **reprodutibilidade** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **reprodutibilidade** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Proveniência e integridade dos artefatos

O notebook grava em **`outputs/`**. A saída CSV é uma tabela de pontuações, não um modelo serializado. Para reproduzir um resultado, registre caminho do dataset, hash, versões, corte temporal, parâmetros e código. Não sobrescreva os dados de entrada nem use arquivos pickle de origem desconhecida.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Hash do dataset de entrada: elemento do manifesto de reprodutibilidade
import hashlib
h = hashlib.sha256()
with (DATA/'ratings.csv').open('rb') as f:
    for bloco in iter(lambda: f.read(1<<20), b''):
        h.update(bloco)
print('SHA-256 ratings.csv:',h.hexdigest())
print('Artefato produzido:',path.as_posix())

SHA-256 ratings.csv: aa289ca83157595d0df6aea1be6a4ded676ddc4385472e8313a8ed9805352646
Artefato produzido: outputs/ranking_popularidade.csv


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

<!-- EP_FASE2_AREA09_LOTE2 -->
### Auditoria complementar — reprodução byte a byte e linhagem

O exemplo já grava o ranking e calcula o hash de `ratings.csv`. Para testar se a saída é efetivamente reprodutível **no mesmo ambiente**, recalculamos o ranking a partir da mesma entrada e com a mesma regularização. Comparamos o CSV gerado com a serialização recalculada **byte a byte**, verificamos seu conteúdo lógico e registramos hashes de entrada/saída em um manifesto adicional, sem carimbar horário.

**Limites:** igualdade de hashes demonstra repetibilidade local, **não** garante resultados idênticos entre versões de bibliotecas, sistemas operacionais ou plataformas. O ranking é ajustado à base integral e não representa avaliação fora da amostra. O manifesto não contém segredos, dados pessoais ou identificadores individuais de usuários.


In [6]:
# Checagem de reprodução com os mesmos dados, parâmetros e ambiente de execução.
import hashlib
import json
import platform
import numpy as np
ep15_input_path = DATA / 'ratings.csv'
ep15_output_path = out / 'ranking_popularidade.csv'
ep15_recalculado = fit_popularity(ratings, reg=20).rename('score')
ep15_bytes_recalculados = ep15_recalculado.to_csv().encode('utf-8')
ep15_bytes_salvos = ep15_output_path.read_bytes()
ep15_sha_recalculado = hashlib.sha256(ep15_bytes_recalculados).hexdigest()
ep15_sha_saida = hashlib.sha256(ep15_bytes_salvos).hexdigest()
ep15_sha_entrada = hashlib.sha256(ep15_input_path.read_bytes()).hexdigest()
assert ep15_sha_recalculado == ep15_sha_saida, 'O CSV persistido difere da recomputação local.'
ep15_lido = pd.read_csv(ep15_output_path).set_index('movieId')['score']
pd.testing.assert_series_equal(
    ep15_lido.sort_index(), ep15_recalculado.sort_index(),
    check_names=True, rtol=1e-12, atol=1e-12
)
ep15_manifesto = {
    'entrada': 'datasets/ratings.csv', 'sha256_entrada': ep15_sha_entrada,
    'saida': 'outputs/ranking_popularidade.csv', 'sha256_saida': ep15_sha_saida,
    'recomputacao_igual_byte_a_byte': True, 'n_itens_rankeados': int(len(ep15_lido)),
    'metodo': 'media_suavizada', 'regularizacao': 20,
    'python': platform.python_version(), 'pandas': pd.__version__,
    'numpy': np.__version__,
    'escopo': 'reprodutibilidade local, sem avaliacao fora da amostra',
}
(out / '15_verificacao_reprodutibilidade.json').write_text(
    json.dumps(ep15_manifesto, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print('Recomputação idêntica byte a byte:', ep15_sha_recalculado == ep15_sha_saida)
print('SHA-256 da saída:', ep15_sha_saida)
print('Manifesto adicional:', out / '15_verificacao_reprodutibilidade.json')


Recomputação idêntica byte a byte: True
SHA-256 da saída: 2370eb44696e218bed104768771cf198708db6c48c8c36ee9e89a45473fb77f4
Manifesto adicional: outputs/15_verificacao_reprodutibilidade.json


## Limitações e boas práticas
Saídas ficam em `outputs/` (ignoradas pelo Git). Em produção, registre linhagem e controle acesso aos dados.

## Exercícios
Descreva como verificar se um artefato foi gerado com o mesmo dataset.

## Soluções comentadas
Registre hash SHA-256 do arquivo de entrada, hiperparâmetros, versões, partição e data.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [7]:
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
import hashlib
arquivo_dados = DATA / "ratings.csv"
sha = hashlib.sha256(arquivo_dados.read_bytes()).hexdigest()
import json as _json
(out / "15_manifesto_ranking.json").write_text(_json.dumps({
    "arquivo_entrada": "datasets/ratings.csv", "sha256_entrada": sha,
    "arquivo_saida": "outputs/ranking_popularidade.csv", "metodo": "media_suavizada",
    "regularizacao": 20, "observacao": "Baseline descritivo ajustado na base integral; nao e avaliacao fora da amostra"
}, indent=2, ensure_ascii=False) + chr(10), encoding="utf-8")
print('Outputs do capítulo 15 salvos em:', out.resolve())


Outputs do capítulo 15 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
